In [8]:
from trainer import RnnlmTrainer
from common.util import eval_perplexity
from common.optimizer import SGD
from model.rnnlm import BetterRnnlm
from data import load_dataset

In [9]:
batch_size = 20
wordvec_size = 1024
hidden_size = 1024
time_size = 60
lr = 20.0
max_epoch = 40
max_grad = 0.25
dropout = 0.5

In [ ]:
corpus = load_dataset("./data/filtered")
corpus_val = corpus
corpus_test = corpus

In [ ]:
print(type(corpus), corpus.get_concated_data().shape)

In [5]:
vocab_size = corpus.tokenizer.vocab_size
xs = corpus.get_concated_data()[:-1]
ts = corpus.get_concated_data()[1:]

In [10]:
model = BetterRnnlm(vocab_size, wordvec_size, hidden_size, dropout)
optimizer = SGD(lr)
trainer = RnnlmTrainer(model, optimizer)

In [ ]:
best_ppl = float('inf')
for epoch in range(max_epoch)[1:]:
    trainer.fit(xs, ts, max_epoch=1, batch_size=batch_size,
                time_size=time_size, max_grad=max_grad,
                dataloader=corpus)

    model.reset_state()
    ppl = eval_perplexity(model, corpus.get_concated_data())
    print('검증 퍼플렉서티: ', ppl)

    if best_ppl > ppl:
        best_ppl = ppl
        model.save_params()
    else:
        lr /= 4.0
        optimizer.lr = lr

    model.reset_state()
    print('-' * 50)
    model.save_params(f"Korean_essay_{epoch}")

In [12]:
model.save_params("Korean_essay.pkl")

In [ ]:
# 테스트 데이터로 평가
model.reset_state()
ppl_test = eval_perplexity(model, corpus_test)
print('테스트 퍼플렉서티: ', ppl_test)